In [9]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [1]:
import pandas as pd
df = pd.read_csv("../data/processed/kidney_clean.csv")
df.head()

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38.0,6000.0,4.8,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31.0,7500.0,4.8,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [2]:
binary_map = {"yes": 1, "no": 0, "good": 1, "poor": 0, "normal": 1, "abnormal": 0,
              "present": 1, "notpresent": 0, "ckd": 1, "notckd": 0}
cat_cols = ["htn", "dm", "cad", "appet", "pe", "ane", "rbc", "pc", "pcc", "ba", "classification"]
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower().map(binary_map)
df[cat_cols].apply(lambda x: x.unique())

htn               [1.0, 0.0, nan]
dm                [1.0, 0.0, nan]
cad               [0.0, 1.0, nan]
appet             [1.0, 0.0, nan]
pe                [0.0, 1.0, nan]
ane               [0.0, 1.0, nan]
rbc               [nan, 1.0, 0.0]
pc                [1.0, 0.0, nan]
pcc               [0.0, 1.0, nan]
ba                [0.0, 1.0, nan]
classification             [1, 0]
dtype: object

In [3]:
from sklearn.preprocessing import StandardScaler
numeric_cols = ["age","bp","sg","al","su","bgr","bu","sc","sod","pot","hemo","pcv","wc","rc"]
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
df[numeric_cols].describe().loc[["mean","std"]]

,age,bp,sg,al,su,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc
mean,0.000000,-2.309264e-16,3.183231e-14,3.552714e-17,-7.105427e-17,0.000000,-1.065814e-16,7.105427e-17,6.039613e-16,-5.329071e-17,-7.815970e-16,-4.263256e-16,-1.776357e-17,-2.486900e-16
std,1.001252,1.001252e+00,1.001252e+00,1.001252e+00,1.001252e+00,1.001252,1.001252e+00,1.001252e+00,1.001252e+00,1.001252e+00,1.001252e+00,1.001252e+00,1.001252e+00,1.001252e+00


In [4]:
df["bun_creatinine_ratio"] = df["bu"] / df["sc"]
df["anemia_ckd_flag"] = ((df["ane"] == 1) & (df["hemo"] < -0.5)).astype(int)
df[["bun_creatinine_ratio","anemia_ckd_flag"]].describe()

,bun_creatinine_ratio,anemia_ckd_flag
count,400.000000,400.000000
mean,5.750471,0.125000
std,70.915553,0.331133
min,-66.064562,0.000000
25%,0.678045,0.000000
50%,1.456609,0.000000
75%,2.045918,0.000000
max,1280.548362,1.000000


In [5]:
corr = df[numeric_cols].corr().abs()
import numpy as np
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.9)]
to_drop

[]

In [6]:
df.to_csv("../data/processed/kidney_features.csv", index=False)
import joblib
joblib.dump(scaler, "../src/scaler.joblib")

['../src/scaler.joblib']

In [10]:
from src.features import build_features
df_check, scaler_check = build_features(pd.read_csv("../data/processed/kidney_clean.csv"))
pd.testing.assert_frame_equal(df_check.reset_index(drop=True), df.reset_index(drop=True))